# FER-CNN — all experiments (Google Colab, GPU)

Clones the repo, downloads FER-2013 and FANE from Kaggle, then runs every
experiment of the study and archives each one in its own Drive folder.

Four configurations, each trained at batch size 32 and at 64, so eight runs:

| run | architecture | warm-up | part that gets trained |
|---|---|---|---|
| `custom_cnn_bs32` / `bs64` | ResNet-style CNN | — | the whole network, from scratch |
| `xception_1phase_bs32` / `bs64` | Xception | no | from `block14` on |
| `xception_2phase_bs32` / `bs64` | Xception | 5 epochs | from `block14` on |
| `xception_full_bs32` / `bs64` | Xception | 5 epochs | the whole base |

`1phase` against `2phase` isolates the warm-up, `2phase` against `full`
isolates how deep the fine-tuning goes, and the two batch sizes are the same
comparison run twice.

Everything ends up on Drive like this:

| folder on Drive | what it contains |
|---|---|
| `eda/` | dataset figures and class counts |
| `runs/<run name>/` | one folder per run, with checkpoint, metrics and confusion matrices |
| `final/` | the tables and figures that compare all the runs, plus the two winning checkpoints |

**This takes several hours and Colab will probably disconnect before the end.**
That is fine, every finished run is copied to Drive straight away and is
restored instead of retrained, so just run the notebook again from the top and
it will pick up where it stopped.

## 1. Repository and dependencies

In [ ]:
REPO_URL = 'https://github.com/xydani/fer-cnn.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

!git clone --branch {BRANCH} {REPO_URL} fer-cnn
%cd fer-cnn

In [ ]:
!pip install -q -r requirements.txt

## 2. Datasets

In [ ]:
import getpass
import json
import os

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!pip install -q kaggle

In [ ]:
!mkdir -p data/raw/fer-2013 data/raw/fane_data
!kaggle datasets download -d msambare/fer2013 -p data/raw/fer-2013 --unzip
!kaggle datasets download -d furcifer/fane-facial-expressions-and-emotion-dataset -p data/raw/fane_data --unzip

## 3. Drive and helpers

Each run writes to the usual project folders and is then copied into its own
folder on Drive, so no run can overwrite another one and a lost session only
costs the run that was in progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = '/content/drive/MyDrive/fer-cnn-results'  #@param {type:"string"}
EPOCHS = 100  #@param {type:"integer"}
WARMUP_EPOCHS = 5  #@param {type:"integer"}
BATCH_SIZES = [32, 64]

DRIVE_ROOT = Path(DRIVE_DIR)
DRIVE_RUNS = DRIVE_ROOT / 'runs'
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)

METRICS = Path('results/metrics')
FIGURES = Path('results/figures')
CHECKPOINTS = Path('models_saved')
for folder in (METRICS, FIGURES, CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)


def run(*args):
    print('$ python', *args, flush=True)
    process = subprocess.Popen(
        [sys.executable, *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'{args[0]} failed with exit code {code}, see the output above')


def archive_results(name):
    dest = DRIVE_ROOT / name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree('results', dest / 'results', dirs_exist_ok=True)
    print(f'saved to {dest}')


def archive_run(name):
    dest = DRIVE_RUNS / name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CHECKPOINTS / f'{name}.keras', dest / f'{name}.keras')
    for folder in (METRICS, FIGURES):
        for path in folder.glob(f'{name}_*'):
            shutil.copy2(path, dest / path.name)
    print(f'saved {name} to {dest}')


def restore_run(name):
    source = DRIVE_RUNS / name
    checkpoint = source / f'{name}.keras'
    if not checkpoint.exists():
        return False
    shutil.copy2(checkpoint, CHECKPOINTS / f'{name}.keras')
    for path in source.glob(f'{name}_*'):
        shutil.copy2(path, (METRICS if path.suffix == '.json' else FIGURES) / path.name)
    return True


def experiment(family, model, batch_size, warmup=0, fine_tune_from=None):
    name = f'{family}_bs{batch_size}'
    if restore_run(name):
        print(f'{name} is already on Drive, restored without retraining\n')
        return
    arguments = ['src/train.py', '--model', model, '--run_name', name,
                 '--epochs', str(EPOCHS), '--batch_size', str(batch_size),
                 '--warmup_epochs', str(warmup)]
    if fine_tune_from is not None:
        arguments += ['--fine_tune_from', fine_tune_from]
    run(*arguments)
    run('src/evaluate.py', '--runs', name)
    archive_run(name)

## 4. Exploratory analysis

Class counts and sample grids.

In [ ]:
run('src/eda.py')
archive_results('eda')

## 5. Experiments

Every cell below trains the same configuration twice, once per batch size. A
cell that has already finished in an earlier session restores its runs from
Drive in a few seconds instead of training them again.

### 5.1 Custom CNN, trained from scratch

In [ ]:
for batch_size in BATCH_SIZES:
    experiment('custom_cnn', 'custom_cnn', batch_size)

### 5.2 Xception, single phase

The tail is unfrozen from the first epoch, with the head still random.

In [ ]:
for batch_size in BATCH_SIZES:
    experiment('xception_1phase', 'xception', batch_size, warmup=0)

### 5.3 Xception, two phases

The base stays frozen for the first `WARMUP_EPOCHS` epochs so the head can
settle, then `block14` is unfrozen and training continues at the lower rate.

In [ ]:
for batch_size in BATCH_SIZES:
    experiment('xception_2phase', 'xception', batch_size, warmup=WARMUP_EPOCHS)

### 5.4 Xception, full fine-tuning

Same warm-up as above, but the second phase unfreezes the whole base instead of
just `block14`. The BatchNorm layers stay frozen anyway, recomputing their
statistics on batches this small is the usual way to ruin pretrained weights.
The learning rate is the same 1e-4 as the partial fine-tuning, so the number of
trainable layers is the only thing that changes between the two.

In [ ]:
for batch_size in BATCH_SIZES:
    experiment('xception_full', 'xception', batch_size,
               warmup=WARMUP_EPOCHS, fine_tune_from='all')

## 6. Final comparison

Scores every run on both test sets, builds the cross-run tables and figures,
then picks the best run of each architecture on validation loss and copies it
under the plain model name, which is what the report figures refer to. The test
sets play no part in the choice.

In [ ]:
import pandas as pd

for model in ('custom_cnn', 'xception'):
    (CHECKPOINTS / f'{model}.keras').unlink(missing_ok=True)
    (METRICS / f'{model}_history.json').unlink(missing_ok=True)

run('src/evaluate.py')
run('src/compare_runs.py')

all_runs = pd.read_csv(METRICS / 'all_runs.csv')
winners = all_runs.sort_values('best_val_loss').groupby('model', as_index=False).first()
print()
print(winners[['model', 'run', 'batch_size', 'best_val_loss']].to_string(index=False))

for _, winner in winners.iterrows():
    shutil.copy2(CHECKPOINTS / f"{winner['run']}.keras", CHECKPOINTS / f"{winner['model']}.keras")
    shutil.copy2(METRICS / f"{winner['run']}_history.json",
                 METRICS / f"{winner['model']}_history.json")

run('src/evaluate.py', '--runs', 'custom_cnn', 'xception')
run('src/plot_results.py', '--runs', 'custom_cnn', 'xception',
    '--epoch_budget_run', 'custom_cnn_bs64')

archive_results('final')
for _, winner in winners.iterrows():
    shutil.copy2(CHECKPOINTS / f"{winner['model']}.keras",
                 DRIVE_ROOT / 'final' / f"{winner['model']}.keras")

## 7. Results

Everything shown here is also in `final/` on Drive.

In [ ]:
from IPython.display import Image, display

for table in ('all_runs.csv', 'batch_size_effect.csv', 'generalization_gap.csv'):
    print(table)
    display(pd.read_csv(METRICS / table))

for figure in ('training_curves.png', 'generalization_gap_slope.png',
               'batch_size_comparison.png', 'finetune_strategy.png',
               'xception_strategy_curves.png'):
    display(Image(str(FIGURES / figure)))